# Software FJ, Fase 4
Universidad Nacional Abierta y a Distancia UNAD

Programación (213023), grupo 213023_285

Tutor Fabián Augusto Ávila Orjuela

Sistema orientado a objetos para clientes, servicios y reservas, sin base de datos y con manejo avanzado de excepciones. Solo usa la biblioteca estándar. Se ejecuta con Restart and run all y cada run crea de nuevo `software_fj.log`.

## 1. Importaciones y constantes

In [1]:
import logging
import math
from abc import ABC, abstractmethod
from datetime import datetime

TASA_IVA = 0.19             # float, IVA del 19 %
HORAS_MAXIMAS_RESERVA = 12  # int, tope de horas por reserva
RUTA_LOG = "software_fj.log"

# Categorías de resultado de cada simulación
OK = "OK"
FALLO = "FALLO CONTROLADO"
INESPERADO = "ERROR INESPERADO"

## 2. Sistema de logging
El log se escribe en un archivo. INFO registra lo que sale bien, WARNING los fallos de datos o de reglas que el usuario puede corregir y ERROR los fallos graves o imprevistos, estos con traceback. No se guardan documentos, correos ni teléfonos.

In [2]:
logging.basicConfig(
    filename=RUTA_LOG,
    filemode="w",  # cada corrida deja un log nuevo
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(funcName)-22s | %(message)s",
    datefmt="%d/%m/%Y %H:%M:%S",
    encoding="utf-8",
    force=True,  # evita líneas repetidas si se ejecuta de nuevo
)
logger = logging.getLogger("software_fj")
logger.info("Sistema Software FJ iniciado")
print("Logging activo en", RUTA_LOG)

Logging activo en software_fj.log


## 3. Jerarquía de excepciones
Todas heredan de `ErrorSoftwareFJ`, así que un solo `except ErrorSoftwareFJ` atrapa cualquier fallo previsto. Cada clase tiene un `codigo` y el `nivel` con que se registra en el log.

In [3]:
class ErrorSoftwareFJ(Exception):
    """Base de las excepciones del sistema."""

    codigo = "SFJ000"
    nivel = logging.WARNING

    def __init__(self, mensaje):
        super().__init__(mensaje)
        self.mensaje = mensaje

    def __str__(self):
        return f"[{self.codigo}] {self.mensaje}"


class DatoInvalidoError(ErrorSoftwareFJ):
    """Dato con formato, tipo o rango incorrecto."""

    codigo = "SFJ101"


class ParametroFaltanteError(ErrorSoftwareFJ):
    """Falta un parámetro obligatorio."""

    codigo = "SFJ102"


class OperacionNoPermitidaError(ErrorSoftwareFJ):
    """La operación no se permite en el estado actual."""

    codigo = "SFJ201"


class EntidadDuplicadaError(OperacionNoPermitidaError):
    """Se intentó registrar una entidad que ya existe."""

    codigo = "SFJ202"


class ReservaInvalidaError(ErrorSoftwareFJ):
    """La reserva no se puede crear ni procesar. Si grave es True se registra como ERROR."""

    codigo = "SFJ301"

    def __init__(self, mensaje, grave=False):
        super().__init__(mensaje)
        if grave:
            self.nivel = logging.ERROR


class ServicioNoDisponibleError(ErrorSoftwareFJ):
    """El servicio no puede atender la solicitud."""

    codigo = "SFJ302"


class CalculoInconsistenteError(ErrorSoftwareFJ):
    """El cálculo de costos dio valores incoherentes."""

    codigo = "SFJ401"
    nivel = logging.ERROR


class EntidadNoEncontradaError(ErrorSoftwareFJ):
    """El identificador no corresponde a ninguna entidad."""

    codigo = "SFJ501"

## 4. Utilidades de validación
Aquí se usa el encadenamiento con `raise ... from`. El error de bajo nivel (`ValueError` o `KeyError`) se convierte en una excepción propia y queda guardado como causa.

In [4]:
def a_numero(valor, campo):
    """Convierte un valor a número finito.

    Recibe el valor y el nombre del campo. Retorna un float.
    El ValueError se encadena a DatoInvalidoError.
    """
    if isinstance(valor, bool):
        raise DatoInvalidoError(f"{campo}: no puede ser verdadero o falso")
    try:
        numero = float(valor)
    except (TypeError, ValueError) as exc:
        raise DatoInvalidoError(f"{campo}: debe ser un número") from exc
    if not math.isfinite(numero):
        raise DatoInvalidoError(f"{campo}: debe ser un número finito")
    return numero


def a_entero(valor, campo):
    """Convierte un valor a entero y rechaza los decimales."""
    numero = a_numero(valor, campo)
    if numero != int(numero):
        raise DatoInvalidoError(f"{campo}: debe ser un número entero")
    return int(numero)


def exigir_texto(valor, campo, minimo=1, maximo=80):
    """Valida un texto y lo retorna sin espacios de más.

    Recibe el valor, el campo y el largo mínimo y máximo.
    """
    if not isinstance(valor, str):
        raise DatoInvalidoError(f"{campo}: debe ser un texto")
    limpio = " ".join(valor.split())
    if limpio == "":
        raise DatoInvalidoError(f"{campo}: no puede estar vacío")
    if not minimo <= len(limpio) <= maximo:
        raise DatoInvalidoError(f"{campo}: debe tener entre {minimo} y {maximo} caracteres")
    return limpio


def validar_horas(valor):
    """Retorna las horas como número. Deben ser mayores que cero."""
    horas = a_numero(valor, "horas")
    if horas <= 0:
        raise DatoInvalidoError("horas: debe ser mayor que cero")
    return horas


def exigir_parametro(parametros, nombre, contexto):
    """Retorna un parámetro obligatorio. El KeyError se encadena a ParametroFaltanteError."""
    try:
        return parametros[nombre]
    except KeyError as exc:
        raise ParametroFaltanteError(f"falta '{nombre}' para {contexto}") from exc


def formato_cop(valor):
    """Escribe un valor en pesos colombianos, con punto como separador de miles."""
    return "$" + f"{valor:,.0f}".replace(",", ".")


print("Utilidades de validación cargadas")

Utilidades de validación cargadas


## 5. Entidad abstracta
Clase abstracta de la que heredan Cliente, Servicio y Reserva. Genera un id único de solo lectura y obliga a cada hija a implementar `describir()`.

In [5]:
class Entidad(ABC):
    """Clase abstracta base. Da un id único de solo lectura y la fecha de creación."""

    contadores = {}  # secuencia de ids por prefijo (CLI, SRV, RES)

    def __init__(self, prefijo):
        Entidad.contadores[prefijo] = Entidad.contadores.get(prefijo, 0) + 1
        self._id = f"{prefijo}{Entidad.contadores[prefijo]:04d}"
        self._fecha_creacion = datetime.now()

    def obtener_id(self):
        return self._id

    def obtener_fecha_creacion(self):
        return self._fecha_creacion

    @abstractmethod
    def describir(self):
        """Cada subclase describe la entidad a su manera."""

    def __str__(self):
        return self.describir()


print("Entidad abstracta definida")

Entidad abstracta definida


## 6. Cliente
Los datos son privados y se acceden con getters y setters que validan. Un cliente inválido no se crea. El documento no tiene setter, porque no debe cambiar.

In [6]:
class Cliente(Entidad):
    """Cliente con datos personales encapsulados y validados."""

    def __init__(self, nombre, correo, telefono, documento):
        self._nombre = self._validar_nombre(nombre)
        self._correo = self._validar_correo(correo)
        self._telefono = self._validar_telefono(telefono)
        self._documento = self._validar_documento(documento)
        super().__init__("CLI")  # el id se genera solo si todo fue válido
        logger.info("Cliente creado con id %s", self.obtener_id())

    def _validar_nombre(self, valor):
        nombre = exigir_texto(valor, "nombre", 3, 60)
        for letra in nombre:
            if not (letra.isalpha() or letra in " '"):
                raise DatoInvalidoError("nombre: solo admite letras, espacios y apóstrofes")
        return nombre

    def _validar_correo(self, valor):
        correo = exigir_texto(valor, "correo", 6, 254).lower()
        valido = correo.count("@") == 1 and " " not in correo and ".." not in correo
        if valido:
            usuario, dominio = correo.split("@")
            valido = usuario != "" and "." in dominio and not dominio.startswith(".") and not dominio.endswith(".")
        if not valido:
            raise DatoInvalidoError("correo: no tiene un formato válido")
        return correo

    def _validar_telefono(self, valor):
        telefono = exigir_texto(valor, "teléfono", 10, 16).replace(" ", "")
        if telefono.startswith("+57"):
            telefono = telefono[3:]
        if not (telefono.isdigit() and len(telefono) == 10 and telefono.startswith("3")):
            raise DatoInvalidoError(
                "teléfono: debe ser un celular colombiano de 10 dígitos que empieza por 3, con +57 opcional"
            )
        return telefono

    def _validar_documento(self, valor):
        documento = exigir_texto(valor, "documento", 6, 14).replace(".", "").replace(" ", "")
        if not (documento.isdigit() and 6 <= len(documento) <= 10):
            raise DatoInvalidoError("documento: debe tener entre 6 y 10 dígitos")
        return documento

    def obtener_nombre(self):
        return self._nombre

    def establecer_nombre(self, valor):
        self._nombre = self._validar_nombre(valor)

    def obtener_correo(self):
        return self._correo

    def establecer_correo(self, valor):
        self._correo = self._validar_correo(valor)

    def obtener_telefono(self):
        return self._telefono

    def establecer_telefono(self, valor):
        self._telefono = self._validar_telefono(valor)

    def obtener_documento(self):
        return self._documento

    def obtener_documento_enmascarado(self):
        """Retorna el documento con todo oculto menos los últimos cuatro dígitos."""
        return "*" * (len(self._documento) - 4) + self._documento[-4:]

    def describir(self):
        return (
            f"Cliente {self.obtener_id()}, {self._nombre}, correo {self._correo}, "
            f"documento {self.obtener_documento_enmascarado()}"
        )


print("Clase Cliente definida")

Clase Cliente definida


## 7. Servicio abstracto y sobrecarga
`Servicio` define lo que todo servicio debe hacer y calcula el costo en un solo lugar. Python no permite dos métodos con el mismo nombre, por eso la sobrecarga de `calcular_costo` se simula con parámetros opcionales, que son `con_iva`, `descuento` (un número o un texto como `"10%"`) y `adicionales`.

El cálculo sigue este orden. Primero la base, luego se suman los adicionales, se resta el descuento y al final se aplica el IVA sobre el valor descontado.

In [7]:
class Servicio(Entidad):
    """Clase abstracta de los servicios que ofrece Software FJ."""

    def __init__(self, nombre, tarifa_hora, disponible=True):
        self.establecer_nombre(nombre)
        self.establecer_tarifa(tarifa_hora)
        self.establecer_disponible(disponible)
        super().__init__("SRV")
        logger.info("Servicio creado con id %s, tipo %s", self.obtener_id(), self.tipo())

    def obtener_nombre(self):
        return self._nombre

    def establecer_nombre(self, valor):
        self._nombre = exigir_texto(valor, "nombre del servicio", 3, 60)

    def obtener_tarifa(self):
        return self._tarifa_hora

    def establecer_tarifa(self, valor):
        tarifa = a_numero(valor, "tarifa por hora")
        if tarifa <= 0:
            raise DatoInvalidoError("tarifa por hora: debe ser mayor que cero")
        self._tarifa_hora = tarifa

    def obtener_disponible(self):
        return self._disponible

    def establecer_disponible(self, valor):
        if not isinstance(valor, bool):
            raise DatoInvalidoError("disponible: debe ser verdadero o falso")
        self._disponible = valor

    # Métodos que cada servicio concreto debe sobrescribir
    @abstractmethod
    def tipo(self):
        """Retorna el nombre del tipo de servicio."""

    @abstractmethod
    def validar_parametros(self, horas, parametros):
        """Valida los parámetros propios del servicio y retorna un diccionario limpio."""

    @abstractmethod
    def calcular_costo_base(self, horas, parametros):
        """Retorna el costo base, sin adicionales, descuento ni IVA."""

    @abstractmethod
    def disponibilidad(self, horas, parametros):
        """Retorna True si el servicio puede atender la solicitud."""

    # Métodos opcionales, solo se sobrescriben si el servicio maneja recursos limitados
    def ocupar(self, horas, parametros):
        """Toma recursos. Por defecto no hace nada."""

    def liberar(self, horas, parametros):
        """Devuelve recursos. Por defecto no hace nada."""

    def desglosar_costo(self, horas, parametros, con_iva=False, descuento=0, adicionales=None):
        """Retorna el costo desglosado en un diccionario.

        Los parámetros opcionales simulan la sobrecarga.
        """
        horas = validar_horas(horas)
        parametros = self.validar_parametros(horas, parametros or {})
        base = self.calcular_costo_base(horas, parametros)
        if base <= 0:
            raise CalculoInconsistenteError("el costo base debe ser mayor que cero")
        extras = self._sumar_adicionales(adicionales)
        porcentaje = a_numero(str(descuento).replace("%", ""), "descuento")
        if not 0 <= porcentaje <= 100:
            raise CalculoInconsistenteError(
                f"el descuento de {porcentaje:g} por ciento queda fuera del rango de 0 a 100"
            )
        bruto = base + extras
        valor_descuento = bruto * porcentaje / 100
        subtotal = bruto - valor_descuento
        iva = subtotal * TASA_IVA if con_iva else 0
        return {
            "base": round(base, 2),
            "adicionales": round(extras, 2),
            "descuento_porcentaje": porcentaje,
            "descuento_valor": round(valor_descuento, 2),
            "subtotal": round(subtotal, 2),
            "iva": round(iva, 2),
            "total": round(subtotal + iva, 2),
        }

    def calcular_costo(self, horas, parametros, con_iva=False, descuento=0, adicionales=None):
        """Retorna solo el total. Es la versión simple de desglosar_costo."""
        return self.desglosar_costo(horas, parametros, con_iva, descuento, adicionales)["total"]

    def _sumar_adicionales(self, adicionales):
        """Suma los cobros opcionales, cada uno una pareja (nombre, valor)."""
        total = 0
        for adicional in adicionales or []:
            try:
                nombre, valor = adicional
            except (TypeError, ValueError) as exc:
                raise DatoInvalidoError("adicional: debe ser una pareja (nombre, valor)") from exc
            exigir_texto(nombre, "nombre del adicional", 2, 40)
            valor = a_numero(valor, "valor del adicional")
            if valor < 0:
                raise DatoInvalidoError("valor del adicional: no puede ser negativo")
            total += valor
        return total


print("Clase abstracta Servicio definida")

Clase abstracta Servicio definida


## 8. Servicios concretos
Los tres heredan de `Servicio` y responden a los mismos métodos, cada uno a su manera, que es el polimorfismo.
* `ReservaSala` exige `asistentes` y respeta el aforo.
* `AlquilerEquipo` exige `unidades` y controla un inventario.
* `Asesoria` cobra según el `nivel` (junior, senior o experto) y controla las horas libres del asesor.

In [8]:
class ReservaSala(Servicio):
    """Sala con aforo máximo."""

    def __init__(self, nombre, tarifa_hora, capacidad, disponible=True):
        capacidad = a_entero(capacidad, "capacidad")
        if capacidad < 1:
            raise DatoInvalidoError("capacidad: debe ser al menos 1")
        self._capacidad = capacidad
        super().__init__(nombre, tarifa_hora, disponible)

    def obtener_capacidad(self):
        return self._capacidad

    def tipo(self):
        return "Reserva de sala"

    def validar_parametros(self, horas, parametros):
        bruto = exigir_parametro(parametros, "asistentes", f"reservar {self.obtener_nombre()}")
        asistentes = a_entero(bruto, "asistentes")
        if asistentes < 1:
            raise DatoInvalidoError("asistentes: debe ser al menos 1")
        if asistentes > self._capacidad:
            raise ServicioNoDisponibleError(
                f"{self.obtener_nombre()}: la sala admite {self._capacidad} personas y se pidieron {asistentes}"
            )
        return {"asistentes": asistentes}

    def calcular_costo_base(self, horas, parametros):
        return self.obtener_tarifa() * horas

    def disponibilidad(self, horas, parametros):
        return self.obtener_disponible()

    def describir(self):
        estado = "disponible" if self.obtener_disponible() else "fuera de servicio"
        return (
            f"{self.tipo()} {self.obtener_id()}, {self.obtener_nombre()}, capacidad {self._capacidad}, "
            f"{formato_cop(self.obtener_tarifa())} por hora, {estado}"
        )


class AlquilerEquipo(Servicio):
    """Equipo con inventario limitado."""

    def __init__(self, nombre, tarifa_hora, stock, disponible=True):
        stock = a_entero(stock, "stock")
        if stock < 1:
            raise DatoInvalidoError("stock: debe ser al menos 1")
        self._stock_total = stock
        self._stock_actual = stock
        super().__init__(nombre, tarifa_hora, disponible)

    def obtener_stock_actual(self):
        return self._stock_actual

    def tipo(self):
        return "Alquiler de equipo"

    def validar_parametros(self, horas, parametros):
        bruto = exigir_parametro(parametros, "unidades", f"alquilar {self.obtener_nombre()}")
        unidades = a_entero(bruto, "unidades")
        if unidades < 1:
            raise DatoInvalidoError("unidades: debe ser al menos 1")
        return {"unidades": unidades}

    def calcular_costo_base(self, horas, parametros):
        return self.obtener_tarifa() * horas * parametros["unidades"]

    def disponibilidad(self, horas, parametros):
        return self.obtener_disponible() and parametros["unidades"] <= self._stock_actual

    def ocupar(self, horas, parametros):
        unidades = parametros["unidades"]
        if unidades > self._stock_actual:
            raise ServicioNoDisponibleError(
                f"{self.obtener_nombre()}: quedan {self._stock_actual} unidades y se pidieron {unidades}"
            )
        self._stock_actual -= unidades
        logger.info("Equipo %s ocupado, quedan %s unidades", self.obtener_id(), self._stock_actual)

    def liberar(self, horas, parametros):
        # El inventario nunca supera el stock original
        self._stock_actual = min(self._stock_total, self._stock_actual + parametros["unidades"])
        logger.info("Equipo %s liberado, quedan %s unidades", self.obtener_id(), self._stock_actual)

    def describir(self):
        return (
            f"{self.tipo()} {self.obtener_id()}, {self.obtener_nombre()}, stock {self._stock_actual} "
            f"de {self._stock_total}, {formato_cop(self.obtener_tarifa())} por hora y unidad"
        )


class Asesoria(Servicio):
    """Asesoría cuya tarifa depende del nivel del asesor."""

    NIVELES = {"junior": 1.0, "senior": 1.5, "experto": 2.0}  # multiplicador de la tarifa base

    def __init__(self, nombre, tarifa_hora, especialidad, horas_libres, disponible=True):
        self._especialidad = exigir_texto(especialidad, "especialidad", 3, 60)
        libres = a_numero(horas_libres, "horas libres")
        if libres <= 0:
            raise DatoInvalidoError("horas libres: debe ser mayor que cero")
        self._horas_libres = libres
        self._horas_totales = libres
        super().__init__(nombre, tarifa_hora, disponible)

    def obtener_horas_libres(self):
        return self._horas_libres

    def tipo(self):
        return "Asesoría especializada"

    def _multiplicador(self, nivel):
        """Retorna el multiplicador del nivel. El KeyError se encadena a DatoInvalidoError."""
        try:
            return self.NIVELES[nivel]
        except KeyError as exc:
            opciones = ", ".join(self.NIVELES)
            raise DatoInvalidoError(f"nivel: debe ser uno de ({opciones})") from exc

    def validar_parametros(self, horas, parametros):
        nivel = str(parametros.get("nivel", "junior")).strip().lower()  # si no se indica, es junior
        self._multiplicador(nivel)
        return {"nivel": nivel}

    def calcular_costo_base(self, horas, parametros):
        return self.obtener_tarifa() * self._multiplicador(parametros["nivel"]) * horas

    def disponibilidad(self, horas, parametros):
        return self.obtener_disponible() and horas <= self._horas_libres

    def ocupar(self, horas, parametros):
        if horas > self._horas_libres:
            raise ServicioNoDisponibleError(
                f"{self.obtener_nombre()}: quedan {self._horas_libres:g} horas libres y se pidieron {horas:g}"
            )
        self._horas_libres -= horas
        logger.info("Asesoría %s ocupada, quedan %s horas", self.obtener_id(), self._horas_libres)

    def liberar(self, horas, parametros):
        self._horas_libres = min(self._horas_totales, self._horas_libres + horas)
        logger.info("Asesoría %s liberada, quedan %s horas", self.obtener_id(), self._horas_libres)

    def describir(self):
        return (
            f"{self.tipo()} {self.obtener_id()}, {self.obtener_nombre()} ({self._especialidad}), "
            f"{self._horas_libres:g} horas libres, tarifa base {formato_cop(self.obtener_tarifa())} por hora"
        )


print("Servicios concretos definidos: ReservaSala, AlquilerEquipo, Asesoria")

Servicios concretos definidos: ReservaSala, AlquilerEquipo, Asesoria


## 9. Reserva y estados
Una reserva pendiente pasa a confirmada o cancelada. La confirmada pasa a procesada o cancelada. Cancelada y procesada son estados finales y cualquier otro cambio lanza `OperacionNoPermitidaError`.
* `cancelar` usa `try/finally`, así la reserva queda cancelada aunque falle la devolución de recursos.
* `procesar` usa `try/except/else/finally` y relanza con `raise ... from` si el cálculo es inconsistente.

In [9]:
class EstadoReserva:
    """Estados posibles de una reserva."""

    PENDIENTE = "pendiente"
    CONFIRMADA = "confirmada"
    CANCELADA = "cancelada"
    PROCESADA = "procesada"


# Transiciones permitidas, lo que no esté aquí se rechaza
TRANSICIONES = {
    EstadoReserva.PENDIENTE: [EstadoReserva.CONFIRMADA, EstadoReserva.CANCELADA],
    EstadoReserva.CONFIRMADA: [EstadoReserva.PROCESADA, EstadoReserva.CANCELADA],
    EstadoReserva.CANCELADA: [],
    EstadoReserva.PROCESADA: [],
}


class Reserva(Entidad):
    """Une un cliente, un servicio y una duración, y controla su estado."""

    def __init__(self, cliente, servicio, horas, parametros=None, descuento=0, adicionales=None, con_iva=True):
        if not isinstance(cliente, Cliente):
            raise ReservaInvalidaError("falta un cliente válido")
        if not isinstance(servicio, Servicio):
            raise ReservaInvalidaError("falta un servicio válido")
        horas = validar_horas(horas)
        if horas > HORAS_MAXIMAS_RESERVA:
            raise ReservaInvalidaError(
                f"una reserva no puede superar {HORAS_MAXIMAS_RESERVA} horas y se pidieron {horas:g}"
            )
        self._cliente = cliente
        self._servicio = servicio
        self._horas = horas
        self._parametros = servicio.validar_parametros(horas, parametros or {})
        self._descuento = descuento
        self._adicionales = adicionales
        self._con_iva = con_iva
        self._estado = EstadoReserva.PENDIENTE
        self._total = None
        super().__init__("RES")
        logger.info(
            "Reserva %s creada para cliente %s y servicio %s",
            self.obtener_id(), cliente.obtener_id(), servicio.obtener_id(),
        )

    def obtener_cliente(self):
        return self._cliente

    def obtener_servicio(self):
        return self._servicio

    def obtener_horas(self):
        return self._horas

    def obtener_estado(self):
        return self._estado

    def obtener_total(self):
        """Retorna el desglose final, que es None hasta que la reserva se procesa."""
        return self._total

    def _exigir_transicion(self, destino, accion):
        actual = self._estado
        if destino in TRANSICIONES[actual]:
            return
        if actual == destino:
            motivo = f"ya está {actual}"
        elif not TRANSICIONES[actual]:
            motivo = f"está {actual} y ese es un estado final"
        else:
            motivo = f"está {actual} y desde ahí no se puede {accion}"
        raise OperacionNoPermitidaError(f"no se permite {accion} {self.obtener_id()} porque {motivo}")

    def _cambiar_estado(self, destino):
        anterior = self._estado
        self._estado = destino
        logger.info("Reserva %s pasó de %s a %s", self.obtener_id(), anterior, destino)

    def calcular_total(self):
        return self._servicio.desglosar_costo(
            self._horas, self._parametros, self._con_iva, self._descuento, self._adicionales
        )

    def confirmar(self):
        """Confirma la reserva si el servicio tiene disponibilidad real y ocupa sus recursos."""
        self._exigir_transicion(EstadoReserva.CONFIRMADA, "confirmar")
        servicio = self._servicio
        if not servicio.obtener_disponible():
            raise ServicioNoDisponibleError(f"{servicio.obtener_nombre()}: está fuera de servicio por ahora")
        if not servicio.disponibilidad(self._horas, self._parametros):
            raise ServicioNoDisponibleError(
                f"{servicio.obtener_nombre()}: no tiene recursos libres para lo solicitado"
            )
        servicio.ocupar(self._horas, self._parametros)
        self._cambiar_estado(EstadoReserva.CONFIRMADA)

    def cancelar(self, motivo="sin motivo indicado"):
        """Cancela la reserva y devuelve los recursos si estaba confirmada."""
        estado_previo = self._estado
        self._exigir_transicion(EstadoReserva.CANCELADA, "cancelar")
        try:
            if estado_previo == EstadoReserva.CONFIRMADA:
                self._servicio.liberar(self._horas, self._parametros)
        finally:
            # Aunque falle la liberación, la reserva queda cancelada
            self._cambiar_estado(EstadoReserva.CANCELADA)
            logger.info("Reserva %s cancelada, %s", self.obtener_id(), motivo)

    def procesar(self):
        """Factura la reserva. Si el cálculo es inconsistente, sigue confirmada."""
        self._exigir_transicion(EstadoReserva.PROCESADA, "procesar")
        try:
            desglose = self.calcular_total()
        except CalculoInconsistenteError as exc:
            raise ReservaInvalidaError(
                f"no se pudo calcular el total de {self.obtener_id()}", grave=True
            ) from exc
        else:
            # Solo se ejecuta si el cálculo salió bien
            self._total = desglose
            self._cambiar_estado(EstadoReserva.PROCESADA)
            return desglose
        finally:
            # Se ejecuta siempre, haya error o no
            logger.info("Intento de procesar %s terminado, estado %s", self.obtener_id(), self._estado)

    def describir(self):
        return (
            f"Reserva {self.obtener_id()}, cliente {self._cliente.obtener_id()}, servicio "
            f"{self._servicio.obtener_id()}, {self._horas:g} horas, estado {self._estado}"
        )


print("Reserva y estados definidos")

Reserva y estados definidos


## 10. Gestor
Guarda clientes, servicios y reservas en listas privadas, no permite duplicados y busca por id. Si el id no existe, el `ValueError` de `list.index` se encadena a `EntidadNoEncontradaError`.

In [10]:
class GestorSoftwareFJ:
    """Administra clientes, servicios y reservas en listas internas."""

    def __init__(self):
        self._clientes = []
        self._servicios = []
        self._reservas = []

    # Las consultas retornan copias para que nadie modifique las listas
    def obtener_clientes(self):
        return tuple(self._clientes)

    def obtener_servicios(self):
        return tuple(self._servicios)

    def obtener_reservas(self):
        return tuple(self._reservas)

    def _buscar_por_id(self, lista, identificador, tipo):
        """Busca una entidad por id. El ValueError se encadena a EntidadNoEncontradaError."""
        ids_existentes = []
        for entidad in lista:
            ids_existentes.append(entidad.obtener_id())
        try:
            posicion = ids_existentes.index(identificador)
        except ValueError as exc:
            raise EntidadNoEncontradaError(f"no existe {tipo} con id {identificador}") from exc
        else:
            return lista[posicion]

    def buscar_cliente(self, identificador):
        return self._buscar_por_id(self._clientes, identificador, "cliente")

    def buscar_servicio(self, identificador):
        return self._buscar_por_id(self._servicios, identificador, "servicio")

    def buscar_reserva(self, identificador):
        return self._buscar_por_id(self._reservas, identificador, "reserva")

    def registrar_cliente(self, cliente):
        """Agrega un cliente y rechaza documentos o correos repetidos."""
        if not isinstance(cliente, Cliente):
            raise DatoInvalidoError("cliente: debe ser un objeto Cliente")
        for existente in self._clientes:
            if existente.obtener_id() == cliente.obtener_id():
                raise EntidadDuplicadaError(f"registrar cliente: {cliente.obtener_id()} ya está registrado")
            if existente.obtener_documento() == cliente.obtener_documento():
                raise EntidadDuplicadaError("registrar cliente: ya existe un cliente con ese documento")
            if existente.obtener_correo() == cliente.obtener_correo():
                raise EntidadDuplicadaError("registrar cliente: ya existe un cliente con ese correo")
        self._clientes.append(cliente)
        logger.info("Cliente %s registrado en el gestor", cliente.obtener_id())
        return cliente

    def crear_cliente(self, nombre, correo, telefono, documento):
        return self.registrar_cliente(Cliente(nombre, correo, telefono, documento))

    def registrar_servicio(self, servicio):
        """Agrega un servicio y rechaza repetir el tipo y el nombre."""
        if not isinstance(servicio, Servicio):
            raise DatoInvalidoError("servicio: debe ser un objeto Servicio")
        for existente in self._servicios:
            if existente.obtener_id() == servicio.obtener_id():
                raise EntidadDuplicadaError(f"registrar servicio: {servicio.obtener_id()} ya está registrado")
            mismo_tipo = existente.tipo() == servicio.tipo()
            mismo_nombre = existente.obtener_nombre().lower() == servicio.obtener_nombre().lower()
            if mismo_tipo and mismo_nombre:
                raise EntidadDuplicadaError("registrar servicio: ya existe un servicio con ese tipo y nombre")
        self._servicios.append(servicio)
        logger.info("Servicio %s registrado en el gestor", servicio.obtener_id())
        return servicio

    def crear_reserva(self, id_cliente, id_servicio, horas, parametros=None, descuento=0,
                      adicionales=None, con_iva=True):
        """Busca el cliente y el servicio por id, crea la reserva y la guarda."""
        cliente = self.buscar_cliente(id_cliente)
        servicio = self.buscar_servicio(id_servicio)
        reserva = Reserva(cliente, servicio, horas, parametros, descuento, adicionales, con_iva)
        self._reservas.append(reserva)
        logger.info("Reserva %s registrada en el gestor", reserva.obtener_id())
        return reserva

    def reservas_por_estado(self):
        conteo = {}
        for reserva in self._reservas:
            estado = reserva.obtener_estado()
            conteo[estado] = conteo.get(estado, 0) + 1
        return conteo

    def ingresos_procesados(self):
        total = 0
        for reserva in self._reservas:
            if reserva.obtener_total():
                total += reserva.obtener_total()["total"]
        return total

    def reporte(self):
        """Retorna el estado general en una sola línea."""
        partes = []
        for estado, cantidad in sorted(self.reservas_por_estado().items()):
            partes.append(f"{cantidad} {estado}")
        estados = ", ".join(partes)
        return (
            f"{len(self._clientes)} clientes, {len(self._servicios)} servicios, "
            f"{len(self._reservas)} reservas ({estados}). "
            f"Ingresos facturados {formato_cop(self.ingresos_procesados())}"
        )


print("Gestor definido")

Gestor definido


## 11. Ejecutor seguro
Toda operación pasa por `ejecutar`, que usa `try/except/else/finally` para que el programa nunca se detenga.
* `except ErrorSoftwareFJ` atrapa los fallos previstos y los registra con el nivel de la excepción.
* `except Exception` atrapa lo que nadie previó y lo registra como ERROR con traceback.
* `else` registra el éxito y `finally` deja siempre la línea de cierre.

In [11]:
class EjecutorSeguro:
    """Ejecuta operaciones sin dejar que un error detenga el programa."""

    def __init__(self):
        self.resultados = []

    def _cadena_de_causas(self, error):
        """Recorre __cause__ y retorna el texto con las causas originales."""
        partes = []
        causa = error.__cause__
        while causa is not None:
            nombre = type(causa).__name__
            partes.append(f"{nombre}: {causa}" if str(causa) else nombre)
            causa = causa.__cause__
        return " | ".join(partes)

    def ejecutar(self, titulo, operacion, esperado=OK):
        """Ejecuta una operación sin argumentos, la registra y retorna su resultado."""
        numero = len(self.resultados) + 1
        etiqueta = f"SIM{numero:02d}"
        logger.info("%s inicio, %s", etiqueta, titulo)
        tipo_error = ""
        causa = ""
        try:
            valor = operacion()
        except ErrorSoftwareFJ as exc:
            categoria = FALLO
            detalle = str(exc)
            tipo_error = type(exc).__name__
            causa = self._cadena_de_causas(exc)
            mensaje = f"{etiqueta} fallo controlado {tipo_error}, {detalle}"
            if causa:
                mensaje += f" | causa original {causa}"
            # El traceback completo solo se guarda en los fallos graves
            logger.log(exc.nivel, mensaje, exc_info=exc.nivel >= logging.ERROR)
        except Exception as exc:
            # Cualquier error que nadie previó
            categoria = INESPERADO
            tipo_error = type(exc).__name__
            detalle = f"{tipo_error}: {exc}"
            logger.error("%s error inesperado %s, la aplicación continúa", etiqueta, detalle, exc_info=True)
        else:
            categoria = OK
            detalle = "operación completa" if valor is None else str(valor)
            logger.info("%s exitosa", etiqueta)
        finally:
            logger.info("%s fin", etiqueta)
        resultado = {
            "numero": numero, "titulo": titulo, "categoria": categoria, "esperado": esperado,
            "detalle": detalle, "tipo_error": tipo_error, "causa": causa,
        }
        self.resultados.append(resultado)
        self._mostrar(resultado)
        return resultado

    def _mostrar(self, r):
        print(f"[SIM {r['numero']:02d}] {r['categoria']:<16} | {r['titulo']}")
        prefijo = f"{r['tipo_error']} | " if r["categoria"] == FALLO else ""
        for i, linea in enumerate(r["detalle"].splitlines()):
            print("   " + (prefijo if i == 0 else "") + linea)
        if r["causa"]:
            print("   causa original |", r["causa"])
        if r["categoria"] != r["esperado"]:
            print("   ATENCIÓN, se esperaba", r["esperado"])

    def resumen(self):
        """Retorna los conteos por categoría y por tipo de error."""
        por_categoria = {OK: 0, FALLO: 0, INESPERADO: 0}
        por_tipo = {}
        coinciden = 0
        for r in self.resultados:
            por_categoria[r["categoria"]] += 1
            if r["categoria"] == r["esperado"]:
                coinciden += 1
            if r["tipo_error"]:
                por_tipo[r["tipo_error"]] = por_tipo.get(r["tipo_error"], 0) + 1
        return {
            "total": len(self.resultados),
            "exitos": por_categoria[OK],
            "fallos_controlados": por_categoria[FALLO],
            "errores_inesperados": por_categoria[INESPERADO],
            "coinciden": coinciden,
            "por_tipo_de_error": por_tipo,
        }


def leer_ultimas_lineas(ruta, cantidad=None):
    """Lee el log desde el disco. El OSError se encadena a ErrorSoftwareFJ."""
    try:
        with open(ruta, encoding="utf-8") as archivo:
            lineas = archivo.read().splitlines()
    except OSError as exc:
        raise ErrorSoftwareFJ(f"no se pudo leer el log {ruta}") from exc
    else:
        return lineas if cantidad is None else lineas[-cantidad:]


print("Ejecutor seguro definido")

Ejecutor seguro definido


## 12. Clases abstractas
`Entidad` y `Servicio` son abstractas, por lo tanto Python debe impedir instanciarlas.

In [12]:
for clase_abstracta in (Entidad, Servicio):
    try:
        clase_abstracta()
    except TypeError as error:
        logger.warning("Clase abstracta no instanciable %s", clase_abstracta.__name__)
        print(f"{clase_abstracta.__name__} es abstracta y Python impide crearla")
        print("   mensaje de Python |", error)

Entidad es abstracta y Python impide crearla
   mensaje de Python | Can't instantiate abstract class Entidad without an implementation for abstract method 'describir'
Servicio es abstracta y Python impide crearla
   mensaje de Python | Can't instantiate abstract class Servicio without an implementation for abstract methods 'calcular_costo_base', 'describir', 'disponibilidad', 'tipo', 'validar_parametros'


## 13. Simulaciones
Son 36 operaciones sobre un mismo gestor, unas válidas y otras inválidas. Cada una pasa por el ejecutor seguro, imprime su resultado y queda en el log. Los datos de los clientes son inventados.

In [13]:
gestor = GestorSoftwareFJ()
ejecutor = EjecutorSeguro()
ids = {}  # id de cada entidad creada, guardado con una clave corta


def registrar_cliente(clave, nombre, correo, telefono, documento):
    cliente = gestor.crear_cliente(nombre, correo, telefono, documento)
    ids[clave] = cliente.obtener_id()
    return cliente.describir()


def registrar_servicio(clave, servicio):
    gestor.registrar_servicio(servicio)
    ids[clave] = servicio.obtener_id()
    return servicio.describir()


def crear_reserva(clave, cliente, servicio, horas, parametros=None, descuento=0, adicionales=None, con_iva=True):
    reserva = gestor.crear_reserva(ids[cliente], ids[servicio], horas, parametros, descuento, adicionales, con_iva)
    ids[clave] = reserva.obtener_id()
    return reserva


def reserva(clave):
    return gestor.buscar_reserva(ids[clave])


print("Gestor, ejecutor y funciones auxiliares listos")

Gestor, ejecutor y funciones auxiliares listos


### 13.1 Clientes
Dos válidos, cuatro que deben fallar (correo, teléfono, nombre y documento repetido) y dos pruebas del setter de teléfono.

In [14]:
def sim_cliente_ana():
    return registrar_cliente("ana", "Ana María Torres", "ana.torres@softwarefj.co", "3105551234", "1012345678")


def sim_cliente_carlos():
    return registrar_cliente("carlos", "Carlos Andrés Rojas", "Carlos.Rojas@Correo.com", "+57 320 555 6789", "80.123.456")


def sim_correo_malo():
    return registrar_cliente("malo", "Laura Pérez", "laura.perez@@correo", "3001112233", "52123456")


def sim_telefono_con_letras():
    return registrar_cliente("malo", "Laura Pérez", "laura.perez@correo.com", "300ABC1234", "52123456")


def sim_nombre_vacio():
    return registrar_cliente("malo", " ", "laura.perez@correo.com", "3001112233", "52123456")


def sim_documento_repetido():
    return registrar_cliente("malo", "Laura Pérez", "laura.perez@correo.com", "3001112233", "1012345678")


def sim_telefono_valido():
    cliente = gestor.buscar_cliente(ids["ana"])
    cliente.establecer_telefono("3201112233")
    return f"El teléfono de {cliente.obtener_id()} cambió y pasó la validación"


def sim_telefono_invalido():
    cliente = gestor.buscar_cliente(ids["ana"])
    cliente.establecer_telefono("12345")
    return "No debería llegar aquí"


ejecutor.ejecutar("Registrar cliente válido, Ana María Torres", sim_cliente_ana)
ejecutor.ejecutar("Cliente válido con +57, puntos en el documento y correo en mayúsculas", sim_cliente_carlos)
ejecutor.ejecutar("Cliente con correo mal formado", sim_correo_malo, FALLO)
ejecutor.ejecutar("Cliente con teléfono que contiene letras", sim_telefono_con_letras, FALLO)
ejecutor.ejecutar("Cliente con nombre vacío", sim_nombre_vacio, FALLO)
ejecutor.ejecutar("Cliente con documento repetido", sim_documento_repetido, FALLO)
ejecutor.ejecutar("Cambiar teléfono con el setter, dato válido", sim_telefono_valido)
ejecutor.ejecutar("Cambiar teléfono con el setter, dato inválido", sim_telefono_invalido, FALLO)

[SIM 01] OK               | Registrar cliente válido, Ana María Torres
   Cliente CLI0001, Ana María Torres, correo ana.torres@softwarefj.co, documento ******5678
[SIM 02] OK               | Cliente válido con +57, puntos en el documento y correo en mayúsculas
   Cliente CLI0002, Carlos Andrés Rojas, correo carlos.rojas@correo.com, documento ****3456
[SIM 03] FALLO CONTROLADO | Cliente con correo mal formado
   DatoInvalidoError | [SFJ101] correo: no tiene un formato válido
[SIM 04] FALLO CONTROLADO | Cliente con teléfono que contiene letras
   DatoInvalidoError | [SFJ101] teléfono: debe ser un celular colombiano de 10 dígitos que empieza por 3, con +57 opcional
[SIM 05] FALLO CONTROLADO | Cliente con nombre vacío
   DatoInvalidoError | [SFJ101] nombre: no puede estar vacío
[SIM 06] FALLO CONTROLADO | Cliente con documento repetido
   EntidadDuplicadaError | [SFJ202] registrar cliente: ya existe un cliente con ese documento
[SIM 07] OK               | Cambiar teléfono con el setter, da

{'numero': 8,
 'titulo': 'Cambiar teléfono con el setter, dato inválido',
 'categoria': 'FALLO CONTROLADO',
 'esperado': 'FALLO CONTROLADO',
 'detalle': '[SFJ101] teléfono: debe tener entre 10 y 16 caracteres',
 'tipo_error': 'DatoInvalidoError',
 'causa': ''}

### 13.2 Servicios
Cuatro servicios válidos (uno fuera de servicio, que se usa más adelante) y tres incorrectos, con precio negativo, nombre repetido y capacidad en cero.

In [15]:
def sim_sala():
    return registrar_servicio("sala", ReservaSala("Sala Innovación", 50000, 10))


def sim_equipo():
    return registrar_servicio("equipo", AlquilerEquipo("Portátil de alto rendimiento", 12000, 3))


def sim_asesoria():
    return registrar_servicio(
        "asesoria", Asesoria("Asesoría en arquitectura de software", 90000, "Arquitectura", 10)
    )


def sim_auditorio():
    return registrar_servicio("auditorio", ReservaSala("Sala Auditorio", 120000, 40, disponible=False))


def sim_precio_negativo():
    return registrar_servicio("malo", AlquilerEquipo("Proyector", -5000, 2))


def sim_servicio_repetido():
    return registrar_servicio("malo", ReservaSala("sala innovación", 55000, 8))


def sim_capacidad_cero():
    return registrar_servicio("malo", ReservaSala("Sala Mini", 30000, 0))


ejecutor.ejecutar("Registrar sala Innovación con capacidad para 10 personas", sim_sala)
ejecutor.ejecutar("Registrar alquiler de portátiles, 3 unidades en inventario", sim_equipo)
ejecutor.ejecutar("Registrar asesoría en arquitectura con 10 horas libres", sim_asesoria)
ejecutor.ejecutar("Registrar sala Auditorio, fuera de servicio por mantenimiento", sim_auditorio)
ejecutor.ejecutar("Servicio con precio negativo", sim_precio_negativo, FALLO)
ejecutor.ejecutar("Servicio repetido, misma sala con el nombre en minúsculas", sim_servicio_repetido, FALLO)
ejecutor.ejecutar("Sala con capacidad cero", sim_capacidad_cero, FALLO)

[SIM 09] OK               | Registrar sala Innovación con capacidad para 10 personas
   Reserva de sala SRV0001, Sala Innovación, capacidad 10, $50.000 por hora, disponible
[SIM 10] OK               | Registrar alquiler de portátiles, 3 unidades en inventario
   Alquiler de equipo SRV0002, Portátil de alto rendimiento, stock 3 de 3, $12.000 por hora y unidad
[SIM 11] OK               | Registrar asesoría en arquitectura con 10 horas libres
   Asesoría especializada SRV0003, Asesoría en arquitectura de software (Arquitectura), 10 horas libres, tarifa base $90.000 por hora
[SIM 12] OK               | Registrar sala Auditorio, fuera de servicio por mantenimiento
   Reserva de sala SRV0004, Sala Auditorio, capacidad 40, $120.000 por hora, fuera de servicio
[SIM 13] FALLO CONTROLADO | Servicio con precio negativo
   DatoInvalidoError | [SFJ101] tarifa por hora: debe ser mayor que cero
[SIM 14] FALLO CONTROLADO | Servicio repetido, misma sala con el nombre en minúsculas
   EntidadDuplicadaEr

{'numero': 15,
 'titulo': 'Sala con capacidad cero',
 'categoria': 'FALLO CONTROLADO',
 'esperado': 'FALLO CONTROLADO',
 'detalle': '[SFJ101] capacidad: debe ser al menos 1',
 'tipo_error': 'DatoInvalidoError',
 'causa': ''}

### 13.3 Costos, sobrecarga y polimorfismo
Seis formas de llamar a `calcular_costo` sobre la misma sala, el mismo método en tres servicios distintos y dos fallos de cálculo.

In [16]:
def sim_sobrecarga():
    sala = gestor.buscar_servicio(ids["sala"])
    parametros = {"asistentes": 6}
    extras = [("Proyector", 20000), ("Refrigerio", 15000)]
    casos = [
        ("Costo base", sala.calcular_costo(2, parametros)),
        ("Con IVA del 19 por ciento", sala.calcular_costo(2, parametros, con_iva=True)),
        ("Con descuento numérico de 10", sala.calcular_costo(2, parametros, descuento=10)),
        ("Con descuento escrito como '10%'", sala.calcular_costo(2, parametros, descuento="10%")),
        ("Con dos adicionales opcionales", sala.calcular_costo(2, parametros, adicionales=extras)),
        ("Adicionales, descuento e IVA juntos",
         sala.calcular_costo(2, parametros, con_iva=True, descuento="10%", adicionales=extras)),
    ]
    lineas = []
    for etiqueta, valor in casos:
        lineas.append(f"{etiqueta:<38} {formato_cop(valor):>12}")
    return "\n".join(lineas)


def sim_polimorfismo():
    casos = [
        (ids["sala"], {"asistentes": 6}),
        (ids["equipo"], {"unidades": 2}),
        (ids["asesoria"], {"nivel": "senior"}),
    ]
    lineas = []
    for id_servicio, parametros in casos:
        servicio = gestor.buscar_servicio(id_servicio)
        costo = servicio.calcular_costo(2, parametros)
        lineas.append(f"{servicio.tipo():<24} 2 horas {formato_cop(costo):>12}")
    return "\n".join(lineas)


def sim_descuento_con_letras():
    gestor.buscar_servicio(ids["sala"]).calcular_costo(2, {"asistentes": 6}, descuento="diez")


def sim_descuento_150():
    gestor.buscar_servicio(ids["sala"]).calcular_costo(2, {"asistentes": 6}, descuento=150)


ejecutor.ejecutar("Sobrecarga de calcular_costo con la sala de 2 horas", sim_sobrecarga)
ejecutor.ejecutar("Polimorfismo con tres servicios y 2 horas", sim_polimorfismo)
ejecutor.ejecutar("Descuento escrito con letras", sim_descuento_con_letras, FALLO)
ejecutor.ejecutar("Cálculo inconsistente, descuento de 150 por ciento", sim_descuento_150, FALLO)

[SIM 16] OK               | Sobrecarga de calcular_costo con la sala de 2 horas
   Costo base                                 $100.000
   Con IVA del 19 por ciento                  $119.000
   Con descuento numérico de 10                $90.000
   Con descuento escrito como '10%'            $90.000
   Con dos adicionales opcionales             $135.000
   Adicionales, descuento e IVA juntos        $144.585
[SIM 17] OK               | Polimorfismo con tres servicios y 2 horas
   Reserva de sala          2 horas     $100.000
   Alquiler de equipo       2 horas      $48.000
   Asesoría especializada   2 horas     $270.000
[SIM 18] FALLO CONTROLADO | Descuento escrito con letras
   DatoInvalidoError | [SFJ101] descuento: debe ser un número
   causa original | ValueError: could not convert string to float: 'diez'
[SIM 19] FALLO CONTROLADO | Cálculo inconsistente, descuento de 150 por ciento
   CalculoInconsistenteError | [SFJ401] el descuento de 150 por ciento queda fuera del rango de 0 a 1

{'numero': 19,
 'titulo': 'Cálculo inconsistente, descuento de 150 por ciento',
 'categoria': 'FALLO CONTROLADO',
 'esperado': 'FALLO CONTROLADO',
 'detalle': '[SFJ401] el descuento de 150 por ciento queda fuera del rango de 0 a 100',
 'tipo_error': 'CalculoInconsistenteError',
 'causa': ''}

### 13.4 Reservas
Tres exitosas y doce que deben fallar. Cubren servicio sin disponibilidad, cancelar una reserva procesada, doble confirmación, descuento imposible y parámetros que faltan.

In [17]:
def sim_ciclo_sala():
    res = crear_reserva(
        "res_sala", "ana", "sala", 2, {"asistentes": 6},
        descuento="10%", adicionales=[("Proyector", 20000), ("Refrigerio", 15000)],
    )
    res.confirmar()
    desglose = res.procesar()
    return f"{res.obtener_id()} procesada, total {formato_cop(desglose['total'])}"


def sim_cero_horas():
    return crear_reserva("x", "ana", "sala", 0, {"asistentes": 3})


def sim_horas_en_letras():
    return crear_reserva("x", "ana", "sala", "dos", {"asistentes": 3})


def sim_treinta_horas():
    return crear_reserva("x", "ana", "sala", 30, {"asistentes": 3})


def sim_capacidad_excedida():
    return crear_reserva("x", "ana", "sala", 2, {"asistentes": 40})


def sim_equipo_sin_unidades():
    return crear_reserva("x", "carlos", "equipo", 2)


def sim_nivel_inexistente():
    return crear_reserva("x", "ana", "asesoria", 2, {"nivel": "maestro"})


def sim_cliente_inexistente():
    return gestor.crear_reserva("CLI9999", ids["sala"], 1, {"asistentes": 2})


def sim_auditorio_fuera_de_servicio():
    res = crear_reserva("res_auditorio", "carlos", "auditorio", 3, {"asistentes": 30})
    res.confirmar()


def sim_stock_insuficiente():
    res = crear_reserva("res_stock", "carlos", "equipo", 2, {"unidades": 5})
    res.confirmar()


def sim_alquiler_y_cancelacion():
    res = crear_reserva("res_equipo", "carlos", "equipo", 4, {"unidades": 2})
    equipo = res.obtener_servicio()
    res.confirmar()
    durante = equipo.obtener_stock_actual()
    res.cancelar("el cliente cambió de planes")
    return (
        f"{res.obtener_id()} cancelada. Inventario durante la reserva {durante}, "
        f"después de cancelar {equipo.obtener_stock_actual()}"
    )


def sim_confirmar_dos_veces():
    res = crear_reserva("res_doble", "ana", "asesoria", 2, {"nivel": "junior"})
    res.confirmar()
    res.confirmar()


def sim_cancelar_procesada():
    reserva("res_sala").cancelar("arrepentimiento tardío")


def sim_procesar_descuento_imposible():
    res = crear_reserva("res_descuento", "carlos", "sala", 1, {"asistentes": 4}, descuento=150)
    res.confirmar()
    res.procesar()


def sim_asesoria_senior_con_iva():
    res = crear_reserva("res_asesoria", "ana", "asesoria", 3, {"nivel": "senior"}, con_iva=True)
    res.confirmar()
    desglose = res.procesar()
    return (
        f"{res.obtener_id()} procesada. Base {formato_cop(desglose['base'])}, "
        f"IVA {formato_cop(desglose['iva'])}, total {formato_cop(desglose['total'])}"
    )


ejecutor.ejecutar("Reserva de sala, ciclo completo hasta procesar", sim_ciclo_sala)
ejecutor.ejecutar("Reserva con cero horas", sim_cero_horas, FALLO)
ejecutor.ejecutar("Reserva con la duración escrita en letras", sim_horas_en_letras, FALLO)
ejecutor.ejecutar("Reserva de 30 horas, supera el máximo permitido", sim_treinta_horas, FALLO)
ejecutor.ejecutar("Reserva con capacidad excedida, 40 personas en una sala de 10", sim_capacidad_excedida, FALLO)
ejecutor.ejecutar("Alquiler de equipo sin indicar las unidades", sim_equipo_sin_unidades, FALLO)
ejecutor.ejecutar("Asesoría con un nivel que no existe", sim_nivel_inexistente, FALLO)
ejecutor.ejecutar("Reserva para un cliente que no existe", sim_cliente_inexistente, FALLO)
ejecutor.ejecutar("Confirmar reserva de un servicio fuera de servicio", sim_auditorio_fuera_de_servicio, FALLO)
ejecutor.ejecutar("Confirmar alquiler de 5 equipos con solo 3 en inventario", sim_stock_insuficiente, FALLO)
ejecutor.ejecutar("Alquiler confirmado y luego cancelado", sim_alquiler_y_cancelacion)
ejecutor.ejecutar("Confirmar dos veces la misma reserva", sim_confirmar_dos_veces, FALLO)
ejecutor.ejecutar("Cancelar una reserva que ya fue procesada", sim_cancelar_procesada, FALLO)
ejecutor.ejecutar("Procesar una reserva con descuento de 150 por ciento", sim_procesar_descuento_imposible, FALLO)
ejecutor.ejecutar("Asesoría nivel senior con IVA, ciclo completo", sim_asesoria_senior_con_iva)

[SIM 20] OK               | Reserva de sala, ciclo completo hasta procesar
   RES0001 procesada, total $144.585
[SIM 21] FALLO CONTROLADO | Reserva con cero horas
   DatoInvalidoError | [SFJ101] horas: debe ser mayor que cero
[SIM 22] FALLO CONTROLADO | Reserva con la duración escrita en letras
   DatoInvalidoError | [SFJ101] horas: debe ser un número
   causa original | ValueError: could not convert string to float: 'dos'
[SIM 23] FALLO CONTROLADO | Reserva de 30 horas, supera el máximo permitido
   ReservaInvalidaError | [SFJ301] una reserva no puede superar 12 horas y se pidieron 30
[SIM 24] FALLO CONTROLADO | Reserva con capacidad excedida, 40 personas en una sala de 10
   ServicioNoDisponibleError | [SFJ302] Sala Innovación: la sala admite 10 personas y se pidieron 40
[SIM 25] FALLO CONTROLADO | Alquiler de equipo sin indicar las unidades
   ParametroFaltanteError | [SFJ102] falta 'unidades' para alquilar Portátil de alto rendimiento
   causa original | KeyError: 'unidades'
[SIM 2

{'numero': 34,
 'titulo': 'Asesoría nivel senior con IVA, ciclo completo',
 'categoria': 'OK',
 'esperado': 'OK',
 'detalle': 'RES0007 procesada. Base $405.000, IVA $76.950, total $481.950',
 'tipo_error': '',
 'causa': ''}

### 13.5 Error imprevisto y reporte final
Pedir la reserva 99 de la tupla produce un `IndexError`, que no es del dominio. El ejecutor lo atrapa, lo registra como ERROR con traceback y el programa sigue.

In [18]:
def sim_error_imprevisto():
    return gestor.obtener_reservas()[99]


ejecutor.ejecutar("Error de programación, acceder a la reserva 99 de la lista", sim_error_imprevisto, INESPERADO)
ejecutor.ejecutar("Reporte final del gestor", gestor.reporte)

[SIM 35] ERROR INESPERADO | Error de programación, acceder a la reserva 99 de la lista
   IndexError: tuple index out of range
[SIM 36] OK               | Reporte final del gestor
   2 clientes, 4 servicios, 7 reservas (1 cancelada, 2 confirmada, 2 pendiente, 2 procesada). Ingresos facturados $626.535


{'numero': 36,
 'titulo': 'Reporte final del gestor',
 'categoria': 'OK',
 'esperado': 'OK',
 'detalle': '2 clientes, 4 servicios, 7 reservas (1 cancelada, 2 confirmada, 2 pendiente, 2 procesada). Ingresos facturados $626.535',
 'tipo_error': '',
 'causa': ''}

## 14. Resumen y lectura del log
Se cuentan los resultados, se comprueba que cada simulación dio lo esperado y se lee `software_fj.log` desde el disco.

In [19]:
resumen = ejecutor.resumen()
print("RESUMEN DE LAS SIMULACIONES")
print(f"  Operaciones ejecutadas        {resumen['total']}")
print(f"  Exitosas                      {resumen['exitos']}")
print(f"  Fallos controlados            {resumen['fallos_controlados']}")
print(f"  Errores inesperados atrapados {resumen['errores_inesperados']}")
print(f"  Coinciden con lo esperado     {resumen['coinciden']} de {resumen['total']}")
print("  Errores por tipo de excepción")
for tipo, cantidad in sorted(resumen["por_tipo_de_error"].items()):
    print(f"    {tipo:<28} {cantidad}")

print()
print("ESTADO DEL SISTEMA")
print(" ", gestor.reporte())

lineas_log = leer_ultimas_lineas(RUTA_LOG)
niveles = {"INFO": 0, "WARNING": 0, "ERROR": 0}
for linea in lineas_log:
    for nivel in niveles:
        if f"| {nivel} " in linea:
            niveles[nivel] += 1
print()
print(f"ARCHIVO {RUTA_LOG}")
print(f"  {len(lineas_log)} líneas. INFO {niveles['INFO']}, WARNING {niveles['WARNING']}, ERROR {niveles['ERROR']}")

print()
print("EJEMPLOS DE ENCADENAMIENTO EN EL LOG")
mostradas = 0
for linea in lineas_log:
    if "causa original" in linea and mostradas < 4:
        print(" ", linea)
        mostradas += 1

print()
print("ÚLTIMAS 12 LÍNEAS DEL LOG")
for linea in leer_ultimas_lineas(RUTA_LOG, 12):
    print(" ", linea)

RESUMEN DE LAS SIMULACIONES
  Operaciones ejecutadas        36
  Exitosas                      13
  Fallos controlados            22
  Errores inesperados atrapados 1
  Coinciden con lo esperado     36 de 36
  Errores por tipo de excepción
    CalculoInconsistenteError    1
    DatoInvalidoError            10
    EntidadDuplicadaError        2
    EntidadNoEncontradaError     1
    IndexError                   1
    OperacionNoPermitidaError    2
    ParametroFaltanteError       1
    ReservaInvalidaError         2
    ServicioNoDisponibleError    3

ESTADO DEL SISTEMA
  2 clientes, 4 servicios, 7 reservas (1 cancelada, 2 confirmada, 2 pendiente, 2 procesada). Ingresos facturados $626.535

ARCHIVO software_fj.log
  205 líneas. INFO 130, WARNING 22, ERROR 3

EJEMPLOS DE ENCADENAMIENTO EN EL LOG
  11/11/2026 15:06:09 | WARNING  | ejecutar               | SIM18 fallo controlado DatoInvalidoError, [SFJ101] descuento: debe ser un número | causa original ValueError: could not convert string 

## Notas de diseño
El Anexo 3 no define algunas reglas y se asumieron las siguientes. IVA del 19 %, celular colombiano de 10 dígitos que empieza por 3, documento de 6 a 10 dígitos, máximo de 12 horas por reserva y niveles de asesoría junior, senior y experto con factores 1, 1,5 y 2. Cada una se puede cambiar en el punto donde se define.
* El descuento es un porcentaje, así que `10` y `"10%"` significan lo mismo.
* El id se asigna al final del constructor. Un objeto rechazado por duplicado ya consumió su número, por eso la secuencia puede tener saltos.
* Para conservar el historial del log entre corridas se cambia `filemode="w"` por `"a"` en la celda 2.